# USDA ERS ARMS — Farm Financials, Costs & Production Practices

**What it is.** The **Agricultural Resource Management Survey** — USDA's primary source for
farm-level **economics**: balance sheets, income statements, costs & returns, and production
practices, by commodity and region / farm typology.

| | |
|---|---|
| **Coverage** | United States — national, regions, states (selected), farm types |
| **History** | ~1996 → present (annual) |
| **Cost / key** | **Free** · key via `api.data.gov/signup` (this notebook runs on `DEMO_KEY`) |
| **Docs** | https://www.ers.usda.gov/developer/data-apis/arms-data-api |

**Why it's in Tracera.** Turns agronomy into **dollars** — what does it cost to grow an acre of
corn, and what's the balance sheet of a farm like this? Benchmark a farm's economics, not just
its yield.

In [1]:
# Tracera shared setup — credentials + the ONE sample farm location
# (Every source is queried at this same spot so results are comparable.)
import sys, pathlib, requests, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd()))
from data_sources._common import SAMPLE_FARM, get_key, field_polygon

LAT, LON = SAMPLE_FARM["lat"], SAMPLE_FARM["lon"]
FIPS, STATE, COUNTY = SAMPLE_FARM["fips"], SAMPLE_FARM["state_alpha"], SAMPLE_FARM["county_name"]
print(SAMPLE_FARM["name"], f"| lat={LAT}, lon={LON} | FIPS {FIPS}")

Story County, Iowa (sample farm) | lat=42.05, lon=-93.5 | FIPS 19169


### 1. Connection test — the API works on `DEMO_KEY` (swap in your own for higher limits)

In [2]:
ARMS = "https://api.ers.usda.gov/data/arms"
# Your own key raises rate limits; DEMO_KEY is fine to explore.
ERS_KEY = get_key("ERS_API_KEY", required=False) or "DEMO_KEY"

def arms(endpoint, **params):
    r = requests.get(f"{ARMS}/{endpoint}", params={"api_key": ERS_KEY, **params}, timeout=60)
    r.raise_for_status()
    return r.json()

years = arms("year")["data"]
reports = arms("report")["data"]
print(f"Using {'your ERS key' if ERS_KEY != 'DEMO_KEY' else 'DEMO_KEY'} | "
      f"{len(years)} years available ({min(years)}–{max(years)})")
print("Example reports:", [r["header"] for r in reports])

Using your ERS key | 29 years available (1996–2024)
Example reports: ['Farm Business Balance Sheet', 'Farm Business Income Statement', 'Farm Business Financial Ratios', 'Structural Characteristics', 'Farm Business Debt Repayment Capacity', 'Government Payments', 'Operator Household Income', 'Operator Household Balance Sheet']


### 2. Core query — farm business balance sheet (national), recent year

In [3]:
resp = arms("surveydata", year=2021, report="Farm Business Balance Sheet")
bs = pd.DataFrame(resp["data"])
keep = [c for c in ["stat_Year", "reportName", "category", "categoryValue",
                    "variableName", "variableUnit", "estimate", "median"] if c in bs.columns]
print(f"{len(bs)} rows returned. Sample balance-sheet items:")
bs[keep].head(10)

1152 rows returned. Sample balance-sheet items:


,stat_Year,reportName,category,categoryValue,variableName,variableUnit,estimate,median
0,2021,Farm Business Balance Sheet,Operator Age,34 years or younger,Farms,Number,53592,NaN
1,2021,Farm Business Balance Sheet,Operator Age,35 to 44 years old,Farms,Number,137386,NaN
2,2021,Farm Business Balance Sheet,Operator Age,45 to 54 years old,Farms,Number,232867,NaN
3,2021,Farm Business Balance Sheet,All Farms,TOTAL,Farms,Number,2003754,NaN
4,2021,Farm Business Balance Sheet,Farm Typology,Off-farm occupation farms (2011 to present),Farms,Number,760224,NaN
5,2021,Farm Business Balance Sheet,Collapsed Farm Typology,Residence farms,Farms,Number,988019,NaN
6,2021,Farm Business Balance Sheet,Collapsed Farm Typology,Intermediate farms,Farms,Number,796517,NaN
7,2021,Farm Business Balance Sheet,NASS Region,South region,Farms,Number,255639,NaN
8,2021,Farm Business Balance Sheet,NASS Region,Plains region,Farms,Number,483004,NaN
9,2021,Farm Business Balance Sheet,Farm Resource Region,Prairie Gateway,Farms,Number,295377,NaN


### Notes & how to extend
- **Endpoints:** `year`, `state`, `report`, `category`, `variable` (reference lists) and
  `surveydata` (the values). Filter `surveydata` by `year`, `report`, `variable`, `category`,
  `state`, `farmtype`.
- **Reports:** "Farm Business Balance Sheet", "Farm Business Income Statement", "Structural
  Characteristics", "Cost of Production", etc. (list them via the `report` endpoint).
- **Get your own key:** https://api.data.gov/signup — DEMO_KEY is rate-limited (~30/hour).
- Pair ARMS costs with **QuickStats** yields and **futures/AMS** prices to model per-acre margin.